# 08 - Prediction Regression Pix

Modelo educacional de regressão para prever `valor_total_next_month`.

In [ ]:
from pathlib import Path
import sys

PROJECT_DIR = Path.cwd().resolve().parent if Path.cwd().resolve().name == "notebooks" else Path.cwd().resolve()
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))


In [ ]:
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.sql import Window
from pyspark.sql import functions as F
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from src.config import FIGURES_DIR, PIX_ML_FEATURES_DIR, PIX_REGRESSION_PREDICTIONS_DIR, REPORTS_DIR, create_project_directories
from src.data_quality import ensure_not_empty
from src.spark_session import get_spark_session

create_project_directories(False)
spark = get_spark_session("08-regression-pix")

In [ ]:
features = ["mes_numero", "trimestre", "valor_total_lag_1", "quantidade_transacoes_lag_1", "ticket_medio_lag_1", "crescimento_valor_lag_1", "crescimento_qtd_lag_1", "valor_total_mm3", "quantidade_transacoes_mm3", "ticket_medio_mm3", "flag_crescimento_valor", "flag_crescimento_qtd"]
w = Window.orderBy("ano_mes")
df = (spark.read.parquet(str(PIX_ML_FEATURES_DIR))
      .withColumn("valor_total_next_month", F.lead("valor_total").over(w))
      .dropna(subset=["valor_total_next_month"]))
for col in features:
    df = df.withColumn(col, F.coalesce(F.col(col).cast("double"), F.lit(0.0)))
assembler = VectorAssembler(inputCols=features, outputCol="features")
model_df = assembler.transform(df).select("ano_mes", "features", F.col("valor_total_next_month").alias("label"))
rows = model_df.count()
train_count = max(1, int(rows * 0.75))
train_months = [r["ano_mes"] for r in model_df.orderBy("ano_mes").limit(train_count).select("ano_mes").collect()]
train_df = model_df.filter(F.col("ano_mes").isin(train_months))
test_df = model_df.filter(~F.col("ano_mes").isin(train_months))
model = LinearRegression(featuresCol="features", labelCol="label", predictionCol="prediction", regParam=0.1)
fit_model = model.fit(train_df)
predictions = fit_model.transform(test_df).orderBy("ano_mes")
predictions.write.mode("overwrite").parquet(str(PIX_REGRESSION_PREDICTIONS_DIR))
predictions.show(truncate=False)

In [ ]:
metrics = []
for metric in ["mae", "mse", "rmse", "r2"]:
    evaluator = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName=metric)
    metrics.append((metric.upper() if metric != "r2" else "R2", float(evaluator.evaluate(predictions))))
pd_pred = predictions.select("ano_mes", "label", "prediction").toPandas()
if not pd_pred.empty:
    pd_pred["ape"] = (abs(pd_pred["label"] - pd_pred["prediction"]) / pd_pred["label"]).where(pd_pred["label"] != 0) * 100
    metrics.append(("MAPE", float(pd_pred["ape"].mean())))
metrics_df = spark.createDataFrame(metrics, ["metric", "value"])
metrics_df.toPandas().to_csv(REPORTS_DIR / "regression_metrics.csv", index=False)
metrics_df.show()

plt.figure(figsize=(12, 6))
plt.plot(pd_pred["ano_mes"], pd_pred["label"] / 1_000_000_000, marker="o", label="Real")
plt.plot(pd_pred["ano_mes"], pd_pred["prediction"] / 1_000_000_000, marker="o", label="Predito")
plt.title("Regressão: valor real versus predito")
plt.xlabel("Ano-mês")
plt.ylabel("Valor do próximo mês (R$ bilhões)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.figtext(0.01, 0.01, "Fonte: dados públicos do Banco Central do Brasil | Modelo educacional", fontsize=9)
plt.tight_layout(rect=(0, 0.04, 1, 1))
plt.savefig(FIGURES_DIR / "06_pix_regression_real_vs_predicted.png", dpi=160)
plt.close()

In [ ]:
spark.stop()